# Example 2: Resume Training from Checkpoint
# 기존 체크포인트에서 학습을 이어가는 예제

이 노트북은 저장된 체크포인트에서 학습을 이어가는 방법(fine-tuning)을 보여줍니다.


In [ ]:
import os
import sys

# Add parent directory to path
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

import yaml
import torch
from models import ViT50_3block
from dataloader import create_dataloaders
from training import Trainer
from utils import set_seed, evaluate_model, print_evaluation_summary, plot_results, plot_training_history


## Step 1: Load Configuration
설정 파일을 불러옵니다. Fine-tuning을 위해 낮은 learning rate를 사용합니다.


In [ ]:
# Load configuration
config_path = './config_resume_training.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  - Additional Epochs: {config['training']['num_epochs']}")
print(f"  - Learning Rate (fine-tuning): {config['optimizer']['lr']}")
print(f"  - Checkpoint: {config['checkpoint']['load_path']}")
print(f"  - Batch Size: {config['dataloader']['batch_size']}")


## Step 2: Setup Environment
시드 설정 및 디바이스 설정을 합니다.


In [ ]:
# Set random seed
set_seed(config['seed'])

# Setup device
device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories
os.makedirs(os.path.dirname(config['training']['save_path']), exist_ok=True)
os.makedirs(os.path.dirname(config['evaluation']['save_results_plot']), exist_ok=True)


## Step 3: Load Datasets


In [ ]:
print("Loading datasets...")
train_loader, val_loader, test_loader, lbl_min, lbl_max = create_dataloaders(
    train_path=config['data']['train_path'],
    val_path=config['data']['val_path'],
    test_path=config['data']['test_path'],
    batch_size=config['dataloader']['batch_size'],
    num_workers=config['dataloader']['num_workers'],
    pin_memory=config['dataloader']['pin_memory'],
    seed=config['seed']
)

print(f"✓ Train samples: {len(train_loader.dataset)}")
print(f"✓ Val samples: {len(val_loader.dataset)}")
print(f"✓ Test samples: {len(test_loader.dataset)}")


## Step 4: Create Model and Load Checkpoint
모델을 생성하고 기존 체크포인트를 불러옵니다.


In [ ]:
print("Creating model...")
model = ViT50_3block(
    img_size=config['model']['img_size'],
    patch_size=config['model']['patch_size'],
    embed_dim=config['model']['embed_dim'],
    depth=config['model']['depth'],
    num_heads=config['model']['num_heads'],
    mlp_dim=config['model']['mlp_dim'],
    num_classes=config['model']['num_classes']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Total parameters: {total_params:,}")

# Load checkpoint
print(f"\nLoading checkpoint from: {config['checkpoint']['load_path']}")
checkpoint_path = config['checkpoint']['load_path']
checkpoint = torch.load(checkpoint_path, map_location=device)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)
print(f"✓ Checkpoint loaded successfully")


In [ ]:
print(f"Resuming training for {config['training']['num_epochs']} additional epochs...")
print(f"New learning rate: {config['optimizer']['lr']:.6g}")

trainer = Trainer(model, train_loader, val_loader, config, device)

# Reset early stopping (starting fresh)
trainer.best_val_loss = float('inf')
trainer.wait = 0

# Train
history = trainer.train()


## Step 6: Plot Training History


In [ ]:
plot_training_history(history, save_path=config['evaluation']['save_history_plot'])
print(f"✓ Training history saved to {config['evaluation']['save_history_plot']}")


## Step 7: Evaluate on Test Set


In [ ]:
print("Evaluating on test set...")
model.load_state_dict(torch.load(config['training']['save_path'], map_location=device))
results = evaluate_model(model, test_loader, lbl_min, lbl_max, device)

print_evaluation_summary(results)


## Step 8: Plot Evaluation Results


In [ ]:
plot_results(results, save_path=config['evaluation']['save_results_plot'])
print(f"✓ Evaluation results saved to {config['evaluation']['save_results_plot']}")


## Summary

Resume training이 완료되었습니다!

- **Resumed model**: `./checkpoints/resumed_model.pth`
- **Training history**: `./results/training_history.png`
- **Evaluation results**: `./results/evaluation_results.png`
